# Notebook 1 — Carga y parsing de JSON (EMG-EPN-612)

Este notebook está preparado para **Google Colab** y realiza:
1. Montaje de Google Drive.
2. Importación del módulo `src.data_loader`.
3. Carga y parsing estructurado de todo el dataset a un DataFrame Pandas.
4. Guardado de metadata y seriales NumPy para la siguiente etapa de Features.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print('Entorno base listo ✅')

In [ ]:
# ==============================
# Montaje de Drive (Solo para Colab)
# ==============================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    # CAMBIA ESTO A TU RUTA REAL EN DRIVE
    # El usuario confirmó que está en MyDrive/emg-classification-knn-svm-ann
    PROJECT_PATH = '/content/drive/MyDrive/emg-classification-knn-svm-ann'
    os.chdir(PROJECT_PATH)
    sys.path.append(PROJECT_PATH)
except Exception:
    IN_COLAB = False
    # Si estás en local, el CWD debería ser la raíz del repo
    PROJECT_PATH = os.getcwd()
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)

print(f'IN_COLAB={IN_COLAB}\nDirectorio de trabajo: {os.getcwd()}')

In [ ]:
# ==============================
# Importar Módulos Propios (src/)
# ==============================
%load_ext autoreload
%autoreload 2

from src.config import Config
from src.data_loader import load_all_users

print(f'Ruta TRAIN_JSON_DIR esperada: {Config.TRAIN_JSON_DIR}')
print(f'Ruta TEST_JSON_DIR esperada: {Config.TEST_JSON_DIR}')

In [ ]:
# ==============================
# Carga de Datos
# ==============================
# Puedes usar max_users=3 para una prueba rápida
print("Cargando usuarios de training...")
df_train = load_all_users(Config.TRAIN_JSON_DIR, max_users=None) 

print("Cargando usuarios de testing...")
df_test = load_all_users(Config.TEST_JSON_DIR, max_users=None)

df_samples = pd.concat([df_train, df_test], ignore_index=True)
print(f'Total de muestras parseadas: {len(df_samples)}')
display(df_samples.head(3))

In [ ]:
# Extraer las variables principales
X_signals_raw = df_samples['signal_raw'].to_list()  # lista de np.ndarray [T, C]
y_gesture = df_samples['gesture_name'].to_numpy()
user_ids = df_samples['user_id'].to_numpy()
splits = df_samples['split'].to_numpy()
gt_indexes = df_samples['ground_truth_index'].to_list()

print('Ejemplo X_signals_raw[0].shape (Señal entera):', X_signals_raw[0].shape)
print('Ejemplo de groundTruthIndex (Inicio y fin del gesto):', gt_indexes[0])
print('y_gesture shape:', y_gesture.shape)
print('Gestos únicos:', sorted(np.unique(y_gesture)))

In [ ]:
# ==============================
# Guardado de Artefactos (Processed)
# ==============================
OUTPUT_DIR = Config.PROCESSED_DIR / 'notebook1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Guardamos Metadata ligera en un CSV
meta_cols = ['user_id', 'split', 'sample_key', 'gesture_name', 'signal_len', 'n_channels']
df_samples[meta_cols].to_csv(OUTPUT_DIR / 'samples_metadata.csv', index=False)

# Guardamos DataFrame completo (incluyendo el array signal_raw) en formato pickle
df_samples.to_pickle(OUTPUT_DIR / 'samples_full.pkl')

print(f'Artefactos guardados con éxito en: {OUTPUT_DIR}')
print(os.listdir(OUTPUT_DIR))